# R2 · Calcul littéral et équations — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/r2_calcul_litteral/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/r2_calcul_litteral/03_notebook.ipynb`, créée par `python tools/start_chapter.py R2`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | R2.1 à R2.12, R2.16 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | R2.22 | Vérifier développements et factorisations avec SymPy | 📦 | ★ | 15 |
|  | R2.27 | Chasse à l'erreur : six erreurs de calcul littéral | 🐛 | ★★ | 20 |
| B | R2.18 | Résoudre ax + b = 0 dans tous les cas · `solve_linear` | 🔨 | ★ | 25 |
|  | R2.19 | Résoudre ax + b ≤ 0 et rendre l'intervalle des solutions · `solve_linear_inequality` | 🔨 | ★★ | 40 |
|  | R2.20 | Résoudre \|x − a\| = c · `solve_abs_equation` | 🔨 | ★★ | 30 |
|  | R2.23 | Résoudre avec `sympy.solve` et lire son résultat | 📦 | ★ | 15 |
|  | R2.28 | Écrire un test pytest pour une fonction qui résout une équation | 🛠️ | ★ | 15 |
| C | R2.21 | Système 2 × 2 par la formule de Cramer · `solve_2x2` | 🔨 | ★★ | 40 |
|  | R2.26 | Un système, deux droites : voir la solution | 🎨 | ★★ | 20 |
|  | R2.24 | Prédire le nombre de solutions de trois systèmes, puis tracer leurs droites | 🔮 | ★ | 15 |
|  | R2.25 | Expérience : un système presque dégénéré, un coefficient bouge de 0,001 et la solution s'envole | 🔬 | ★★ | 20 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R2")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : développer, factoriser, résoudre équations, inéquations, valeurs absolues et systèmes, mettre en équation.
- Écrire les quatre fonctions de `mymath/algebra.py` et les tester.
- Vérifier un calcul avec SymPy et savoir lire sa réponse ; voir les droites d'un système, et un système presque dégénéré.

**Rappel express.** $(a \pm b)^2 = a^2 \pm 2ab + b^2$ et $(a + b)(a - b) = a^2 - b^2$ ; $ax + b = 0$ a la solution $-\frac{b}{a}$ si $a \ne 0$ ; diviser une inéquation par un négatif retourne le sens ; $|x - a| = c$ donne $a \pm c$ si $c > 0$ ; un système 2 × 2 a une seule solution exactement quand $D = ad - bc \ne 0$.

## Partie 0 · Vérifier tes exercices papier (✏️ R2.1 à R2.12, 🔗 R2.16)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un nombre avec un point décimal, `-1.5` ; une fraction exacte entre guillemets, `"5/7"` ;
- une liste de coefficients, du plus haut degré au terme constant, zéros compris : `[2, 0, -5]` pour $2x^2 - 5$ ;
- une liste de solutions, dans l'ordre que tu veux, où une fraction s'écrit comme un calcul : `[-2/7, 6]` ;
- un intervalle entre guillemets, `"]-1 ; 4]"` ou `"[0 ; +oo["` ; une réunion avec un U ; l'ensemble vide avec le mot vide entre guillemets ;
- le nombre de solutions d'un système par son numéro : 1 pour aucune, 2 pour une seule, 3 pour une infinité.

Les réponses pas encore remplies affichent ⏳.

**Ex R2.1 — Substituer et réduire**

In [ ]:
# Ex R2.1: the values you found on paper
answer_R2_1a = ...  # a) a number
answer_R2_1b = ...  # b) a number
answer_R2_1c = ...  # c) the list [alpha, beta, gamma]
answer_R2_1d = ...  # d) the list [alpha, beta]
answer_R2_1e = ...  # e) a number

for letter, answer in zip("abcde", [answer_R2_1a, answer_R2_1b, answer_R2_1c, answer_R2_1d, answer_R2_1e]):
    wbm.check(f"R2.1{letter}", answer)

**Ex R2.2 — Développer**

In [ ]:
# Ex R2.2: the values you found on paper
answer_R2_2a = ...  # a) the list [alpha, beta]
answer_R2_2b = ...  # b) the list [alpha, beta, gamma]
answer_R2_2c = ...  # c) the list [alpha, beta, gamma, delta]
answer_R2_2d = ...  # d) the list [alpha, beta, gamma]
answer_R2_2e = ...  # e) the list [alpha, beta, gamma]

for letter, answer in zip("abcde", [answer_R2_2a, answer_R2_2b, answer_R2_2c, answer_R2_2d, answer_R2_2e]):
    wbm.check(f"R2.2{letter}", answer)

**Ex R2.3 — Factoriser par un facteur commun**

In [ ]:
# Ex R2.3: the values you found on paper
answer_R2_3a = ...  # a) the list [k, alpha, beta]
answer_R2_3b = ...  # b) the list [alpha, beta]
answer_R2_3c = ...  # c) the list [alpha, beta]
answer_R2_3d = ...  # d) the list [alpha, beta]
answer_R2_3e = ...  # e) the list [alpha, beta]
answer_R2_3f = ...  # f) the list [alpha, beta]

for letter, answer in zip("abcdef", [answer_R2_3a, answer_R2_3b, answer_R2_3c, answer_R2_3d, answer_R2_3e, answer_R2_3f]):
    wbm.check(f"R2.3{letter}", answer)

**Ex R2.4 — Identités remarquables dans les deux sens**

In [ ]:
# Ex R2.4: the values you found on paper
answer_R2_4a = ...  # a) the list [alpha, beta, gamma]
answer_R2_4b = ...  # b) the list [alpha, beta, gamma]
answer_R2_4c = ...  # c) the list [alpha, beta, gamma]
answer_R2_4d = ...  # d) a number
answer_R2_4e = ...  # e) the list [alpha, beta]
answer_R2_4f = ...  # f) the list [alpha, beta]
answer_R2_4g = ...  # g) an integer
answer_R2_4h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_R2_4a, answer_R2_4b, answer_R2_4c, answer_R2_4d, answer_R2_4e, answer_R2_4f, answer_R2_4g, answer_R2_4h]):
    wbm.check(f"R2.4{letter}", answer)

**Ex R2.5 — Équations du premier degré**

In [ ]:
# Ex R2.5: the values you found on paper
answer_R2_5a = ...  # a) a number
answer_R2_5b = ...  # b) a number
answer_R2_5c = ...  # c) a number
answer_R2_5d = ...  # d) a number
answer_R2_5e = ...  # e) 1, 2 or 3
answer_R2_5f = ...  # f) 1, 2 or 3

for letter, answer in zip("abcdef", [answer_R2_5a, answer_R2_5b, answer_R2_5c, answer_R2_5d, answer_R2_5e, answer_R2_5f]):
    wbm.check(f"R2.5{letter}", answer)

**Ex R2.6 — Équations produit nul et équations avec des fractions**

In [ ]:
# Ex R2.6: the values you found on paper
answer_R2_6a = ...  # a) a list of solutions
answer_R2_6b = ...  # b) a list of solutions
answer_R2_6c = ...  # c) a list of solutions
answer_R2_6d = ...  # d) a list of solutions
answer_R2_6e = ...  # e) a number
answer_R2_6f = ...  # f) a list of solutions
answer_R2_6g = ...  # g) a list of solutions

for letter, answer in zip("abcdefg", [answer_R2_6a, answer_R2_6b, answer_R2_6c, answer_R2_6d, answer_R2_6e, answer_R2_6f, answer_R2_6g]):
    wbm.check(f"R2.6{letter}", answer)

**Ex R2.7 — Inéquations et intervalles**

In [ ]:
# Ex R2.7: the values you found on paper
answer_R2_7a = ...  # a) an interval, as text
answer_R2_7b = ...  # b) an interval, as text
answer_R2_7c = ...  # c) an interval, as text
answer_R2_7d = ...  # d) an interval, as text
answer_R2_7e = ...  # e) an interval, as text

for letter, answer in zip("abcde", [answer_R2_7a, answer_R2_7b, answer_R2_7c, answer_R2_7d, answer_R2_7e]):
    wbm.check(f"R2.7{letter}", answer)

**Ex R2.8 — Réunion et intersection d'intervalles**

In [ ]:
# Ex R2.8: the values you found on paper
answer_R2_8a = ...  # a) an interval, as text
answer_R2_8b = ...  # b) an interval, as text
answer_R2_8c = ...  # c) an interval, as text
answer_R2_8d = ...  # d) an interval, as text
answer_R2_8e = ...  # e) an interval or a union, as text
answer_R2_8f = ...  # f) an interval, as text

for letter, answer in zip("abcdef", [answer_R2_8a, answer_R2_8b, answer_R2_8c, answer_R2_8d, answer_R2_8e, answer_R2_8f]):
    wbm.check(f"R2.8{letter}", answer)

**Ex R2.9 — Équations et inéquations avec une valeur absolue, et la fonction signe**

In [ ]:
# Ex R2.9: the values you found on paper
answer_R2_9a = ...  # a) a list of solutions
answer_R2_9b = ...  # b) a list of solutions
answer_R2_9c = ...  # c) a list of solutions (empty if there is none)
answer_R2_9d = ...  # d) an interval, as text
answer_R2_9e = ...  # e) an interval or a union, as text
answer_R2_9f = ...  # f) an exact decimal
answer_R2_9g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_R2_9a, answer_R2_9b, answer_R2_9c, answer_R2_9d, answer_R2_9e, answer_R2_9f, answer_R2_9g]):
    wbm.check(f"R2.9{letter}", answer)

**Ex R2.10 — Système 2 × 2 par substitution et par combinaison**

In [ ]:
# Ex R2.10: the values you found on paper
answer_R2_10a = ...  # a) the list [x, y]
answer_R2_10b = ...  # b) the list [x, y]
answer_R2_10c = ...  # c) the list [x, y]
answer_R2_10d = ...  # d) the list [x, y]

for letter, answer in zip("abcd", [answer_R2_10a, answer_R2_10b, answer_R2_10c, answer_R2_10d]):
    wbm.check(f"R2.10{letter}", answer)

**Ex R2.11 — Système 2 × 2 par la formule de Cramer**

In [ ]:
# Ex R2.11: the values you found on paper
answer_R2_11a = ...  # a) a number
answer_R2_11b = ...  # b) the list [x, y]
answer_R2_11c = ...  # c) 1, 2 or 3
answer_R2_11d = ...  # d) 1, 2 or 3

for letter, answer in zip("abcd", [answer_R2_11a, answer_R2_11b, answer_R2_11c, answer_R2_11d]):
    wbm.check(f"R2.11{letter}", answer)

**Ex R2.12 — Mettre en équation : trois problèmes**

In [ ]:
# Ex R2.12: the values you found on paper
answer_R2_12a = ...  # a) a number (cm²)
answer_R2_12b = ...  # b) a number (€)
answer_R2_12c = ...  # c) an integer

for letter, answer in zip("abc", [answer_R2_12a, answer_R2_12b, answer_R2_12c]):
    wbm.check(f"R2.12{letter}", answer)

**Ex R2.16 — Deux mesures, deux inconnues : retrouver a et b dans y = ax + b par un système**

In [ ]:
# Ex R2.16: the values you found on paper
answer_R2_16a = ...  # a) the list [a, b]
answer_R2_16b = ...  # b) a number (°C)
answer_R2_16c = ...  # c) a number (°C)
answer_R2_16d = ...  # d) 1, 2 or 3

for letter, answer in zip("abcd", [answer_R2_16a, answer_R2_16b, answer_R2_16c, answer_R2_16d]):
    wbm.check(f"R2.16{letter}", answer)

## Partie A · Calcul littéral : vérifier avec SymPy, chasser les erreurs

Fiche §1 à §3. SymPy comme outil de vérification de tes calculs papier, et une chasse aux erreurs classiques du calcul littéral.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from wbm.nbtools import run_pytest_group

x, y = sp.symbols("x y")


def as_interval(result):
    """(left_closed, low, high, right_closed), or None, -> the interval text that wbm.check reads."""
    if result is None:
        return "vide"
    left, low, high, right = result
    ends = ["-oo" if v == -math.inf else "+oo" if v == math.inf else repr(float(v)) for v in (low, high)]
    return f"{'[' if left else ']'}{ends[0]} ; {ends[1]}{']' if right else '['}"


def raised(function, *args):
    """The name of the exception that function(*args) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳."""
    try:
        function(*args)
    except NotImplementedError:
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex R2.22 — Vérifier développements et factorisations avec SymPy 📦 ★ ⏱️ 15 min
**Objectif :** développer, tester une égalité et factoriser avec SymPy, pour vérifier un calcul fait à la main.  
**Prérequis :** fiche §2 et §3 ; R0 §4 (SymPy) · **Parcours :** C, E

Les symboles `x` et `y` sont déjà créés (cellule des outils). Trois outils de SymPy :
- `sp.expand(e)` développe et réduit ; `sp.factor(e)` factorise ;
- `sp.Poly(e, x).all_coeffs()` donne la liste des coefficients d'un polynôme en $x$, du plus haut degré au terme constant (comme dans la partie 0) ;
- deux expressions $A$ et $B$ sont égales pour **tout** $x$ exactement quand `sp.expand(A - B)` vaut `0` : c'est une preuve, pas un test sur quelques valeurs.

Avec eux :
- a) développe $(3x - 2)(x^2 + 4x - 1)$, et donne la liste de ses coefficients ;
- b) parmi ces quatre égalités, lesquelles sont vraies pour tout $x$ ? Donne leurs lettres dans un texte.

| | Égalité |
|---|---|
| A | $(x - 3)^2 = x^2 - 9$ |
| B | $(2x + 1)^2 - (2x - 1)^2 = 8x$ |
| C | $(x + 1)^3 = x^3 + 3x^2 + 3x + 1$ |
| D | $(x + 2)(x^2 - 2x + 4) = x^3 + 4$ |

- c) factorise $x^2 + x - 12$, puis donne la liste des valeurs de $x$ qui l'annulent (fiche §4, équation produit nul).

Fais d'abord a) et c) à la main : SymPy est là pour vérifier.

In [ ]:
answer_R2_22a = ...  # a) the list of coefficients
answer_R2_22b = ...  # b) the letters of the true equalities, as text
answer_R2_22c = ...  # c) the list of the values of x that make x² + x − 12 equal to 0

In [ ]:
for letter, answer in zip("abc", [answer_R2_22a, answer_R2_22b, answer_R2_22c]):
    wbm.check(f"R2.22{letter}", answer)

### Ex R2.27 — Chasse à l'erreur : six erreurs de calcul littéral 🐛 ★★ ⏱️ 20 min
**Objectif :** repérer les erreurs classiques de calcul littéral, et les corriger.  
**Prérequis :** fiche §2 à §5, « Les pièges classiques » · **Parcours :** C, E

Un camarade a fait huit calculs. **Sans rien exécuter d'abord**, trouve les six qui sont faux.

| | Calcul |
|---|---|
| A | $(x - 5)^2 = x^2 - 25$ |
| B | $-(2x - 3) = -2x + 3$ |
| C | $(3x)^2 = 3x^2$ |
| D | $2(x + 4) - 3(x - 1) = -x + 5$ |
| E | $(x + 2)(x - 2) = x^2 - 4$ |
| F | $-4x < 10 \iff x < -2{,}5$ |
| G | $3x^2 = 6x \iff x = 2$ |
| H | $\dfrac{x + 1}{2} \times 4 = 2x + 1$ |

- a) Les lettres des calculs faux, dans un texte.
- b) Le bon résultat de D : la liste $[\alpha, \beta]$ de $\alpha x + \beta$.
- c) Le bon ensemble des solutions de l'inéquation de F, en intervalle.
- d) La bonne liste des solutions de l'équation de G.

Puis, si tu veux, confirme avec SymPy (`sp.expand`, `sp.solveset`) : la solution montre comment.

In [ ]:
answer_R2_27a = ...  # a) the letters of the wrong calculations, as text
answer_R2_27b = ...  # b) the list [alpha, beta]
answer_R2_27c = ...  # c) an interval, as text
answer_R2_27d = ...  # d) a list of solutions

In [ ]:
for letter, answer in zip("abcd", [answer_R2_27a, answer_R2_27b, answer_R2_27c, answer_R2_27d]):
    wbm.check(f"R2.27{letter}", answer)

## Partie B · Équations et inéquations

Fiche §4 à §6. Trois fonctions de `mymath/algebra.py` (équation, inéquation, valeur absolue), SymPy et ses pièges, et un test que tu écris toi-même.

### Ex R2.18 — Résoudre ax + b = 0 dans tous les cas · `solve_linear` 🔨 ★ ⏱️ 25 min
**Objectif :** programmer la résolution de ax + b = 0, cas dégénérés compris.  
**Prérequis :** fiche §4, « L'équation ax + b = 0 » ; workbookIA 0A §100.4.2 · **mymath :** `algebra.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en R1). Ouvre ta copie `mon_travail/mymath/algebra.py`, lis la docstring du module **et** celle de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. N'utilise ni SymPy ni NumPy dans `algebra.py` : les tests s'en servent pour vérifier tes résultats.

Écris `solve_linear(a, b)` : les **trois cas** de la fiche (§4). Une solution se rend dans une liste, `[x]` (divise avec `/`, pour obtenir un flottant) ; aucune solution, la liste vide `[]` ; tous les réels, la chaîne `"all"`. Compare `a` (et `b`) à 0 avec `==` : les tests des cas dégénérés utilisent des entiers.

La vérification calcule `solve_linear(-5, 2)`, la longueur de `solve_linear(0, -3)` et `solve_linear(0, 0)`, puis lance les tests.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R2")  # reload your saved file

In [ ]:
with wbm.attempt("R2.18a"):
    wbm.check("R2.18a", mymath.algebra.solve_linear(-5, 2), computed=True)
with wbm.attempt("R2.18b"):
    wbm.check("R2.18b", len(mymath.algebra.solve_linear(0, -3)), computed=True)
with wbm.attempt("R2.18c"):
    wbm.check("R2.18c", mymath.algebra.solve_linear(0, 0), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r2_algebra.py", "test_solve_linear_ and not inequality")

### Ex R2.19 — Résoudre ax + b ≤ 0 et rendre l'intervalle des solutions · `solve_linear_inequality` 🔨 ★★ ⏱️ 40 min
**Objectif :** programmer la résolution d'une inéquation du premier degré, changement de sens et cas dégénérés compris.  
**Prérequis :** R2.18 ; fiche §5 ; workbookIA 0A §100.6.2 (`raise`) · **mymath :** `algebra.py` · **Parcours :** C

Écris `solve_linear_inequality(a, b, sense="<")` : l'ensemble des solutions de $ax + b < 0$ (ou `"<="`, `">"`, `">="`), rendu comme un **intervalle** `(left_closed, low, high, right_closed)`, ou `None` pour l'ensemble vide (la docstring du module explique la convention). Une démarche :
1. un `sense` qui n'est pas l'un des quatre symboles lève une `ValueError` ;
2. si `a == 0`, l'inéquation dit « $b < 0$ » (ou $b \le 0$…), vraie pour tout $x$ (toute la droite) ou jamais (`None`) ;
3. sinon, la borne est $-\frac{b}{a}$, et diviser par un $a$ **négatif** retourne le sens (fiche §5) ;
4. `"<"` et `"<="` donnent $]-\infty \;;\; r[$ ou $]-\infty \;;\; r]$ ; `">"` et `">="` donnent $]r \;;\; +\infty[$ ou $[r \;;\; +\infty[$, avec `math.inf` pour l'infini, toujours ouvert.

La vérification transforme ton tuple en texte avec `as_interval` (cellule des outils) pour `wbm.check` : `solve_linear_inequality(-4, -6, ">=")`, puis `solve_linear_inequality(0, 3, "<=")`.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R2")  # reload your saved file

In [ ]:
with wbm.attempt("R2.19a"):
    wbm.check("R2.19a", as_interval(mymath.algebra.solve_linear_inequality(-4, -6, ">=")), computed=True)
with wbm.attempt("R2.19b"):
    wbm.check("R2.19b", as_interval(mymath.algebra.solve_linear_inequality(0, 3, "<=")), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r2_algebra.py", "test_solve_linear_inequality_")

### Ex R2.20 — Résoudre |x − a| = c · `solve_abs_equation` 🔨 ★★ ⏱️ 30 min
**Objectif :** programmer la résolution de |x − a| = c en lisant la valeur absolue comme une distance.  
**Prérequis :** fiche §6 · **mymath :** `algebra.py` · **Parcours :** C

Écris `solve_abs_equation(a, c)` : les points à distance `c` de `a` (fiche §6, le tableau des cas), rendus dans une liste de flottants **dans l'ordre croissant** : deux solutions, une seule, ou aucune.

La vérification calcule `solve_abs_equation(-1.5, 4)` et `solve_abs_equation(7, 0)`.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R2")  # reload your saved file

In [ ]:
with wbm.attempt("R2.20a"):
    wbm.check("R2.20a", mymath.algebra.solve_abs_equation(-1.5, 4), computed=True)
with wbm.attempt("R2.20b"):
    wbm.check("R2.20b", mymath.algebra.solve_abs_equation(7, 0), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r2_algebra.py", "test_solve_abs_equation_")

### Ex R2.23 — Résoudre avec `sympy.solve` et lire son résultat 📦 ★ ⏱️ 15 min
**Objectif :** résoudre des équations et un système avec SymPy, et savoir lire (et se méfier de) ce qu'il renvoie.  
**Prérequis :** fiche §4 et §7 (les cellules SymPy) · **Parcours :** C

`sp.solve(e, x)` résout « $e = 0$ » et renvoie une **liste** de solutions ; pour une équation $A = B$, on lui donne `A - B` (fiche §4). `sp.solveset(e, x, sp.S.Reals)` renvoie un **ensemble** de SymPy, qui sait dire « aucune solution » (`EmptySet`) et « tous les réels » (`Reals`).

- a) Résous $4x - 7 = x + 5$ avec `sp.solve`, et donne **la solution** (un nombre, pas la liste).
- b) Résous $3(x - 2) = 3x - 6$ avec `sp.solve` : combien d'éléments la liste renvoyée a-t-elle ? Fais de même avec $5(x - 1) = 5x - 2$. Combien de solutions chacune de ces deux équations a-t-elle vraiment (fiche §4) ?
- c) Résous à nouveau $3(x - 2) = 3x - 6$, avec `sp.solveset` sur les réels, et donne l'ensemble obtenu (tu peux le passer tel quel à `wbm.check`).
- d) Résous le système $\begin{cases} 7x - 2y = 3 \\ x + y = 3 \end{cases}$ avec `sp.solve([e1, e2], [x, y])` : le résultat est un **dictionnaire**. Donne la liste $[x, y]$ lue dedans.

In [ ]:
answer_R2_23a = ...  # a) a number
answer_R2_23b = ...  # b) an integer: the length of the list returned for 3(x − 2) = 3x − 6
answer_R2_23c = ...  # c) the set returned by sp.solveset (the SymPy object itself is fine)
answer_R2_23d = ...  # d) the list [x, y]

In [ ]:
for letter, answer in zip("abcd", [answer_R2_23a, answer_R2_23b, answer_R2_23c, answer_R2_23d]):
    wbm.check(f"R2.23{letter}", answer)

### Ex R2.28 — Écrire un test pytest pour une fonction qui résout une équation 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire un test qui vérifie une fonction de résolution, en substituant la solution plutôt qu'en la recopiant.  
**Prérequis :** R2.18 ; fiche §4 · **Parcours :** C

Un **test** appelle une fonction sur des cas dont on connaît la réponse, et échoue (`assert`) si elle se trompe. `pytest` exécute toutes les fonctions `test_...` d'un fichier `tests/test_....py` : c'est ce que lance `run_pytest_group`.

Écris le corps de `check_solve_linear(solve_linear)` : des `assert` qui vérifient la fonction `solve_linear` qu'on lui passe, dans les trois cas de la fiche (§4) :
1. un cas à une solution, **vérifié en substituant** : la liste rendue a un seul élément `s`, et `a * s + b` est presque nul (`abs(...) < 1e-9`). C'est plus sûr que de comparer à une valeur que tu aurais calculée (et peut-être mal) ;
2. un cas sans solution : la liste vide ;
3. un cas où tous les réels sont solutions : la chaîne `"all"`.

La vérification lance ton test sur une version juste (la référence), où il doit **passer**, puis sur trois versions fausses (cellule suivante), où il doit **échouer** à chaque fois : un test qui ne rate aucune erreur ne sert à rien. Dans un fichier de tests, tu écrirais ensuite `def test_solve_linear(): check_solve_linear(mymath.algebra.solve_linear)`.

In [ ]:
# Three faulty versions of solve_linear: a good test catches each one
def solve_linear_wrong_sign(a, b):
    if a == 0:
        return "all" if b == 0 else []
    return [b / a]


def solve_linear_divides_by_zero(a, b):
    return [-b / a]


def solve_linear_forgets_all(a, b):
    if a == 0:
        return []
    return [-b / a]

In [ ]:
def check_solve_linear(solve_linear):
    """Fail (assert) if solve_linear is wrong in one of the three cases of the sheet (§4)."""
    raise NotImplementedError("écris tes assert")

In [ ]:
with wbm.attempt("R2.28"):
    reference = wbm.load_mymath("ref", alias="mymath_ref").algebra.solve_linear
    try:
        check_solve_linear(reference)
    except NotImplementedError:
        raise
    except Exception as exc:  # noqa: BLE001 - a test that fails on a right function
        print(f"❌ Ex R2.28 : ton test échoue sur une fonction juste ({type(exc).__name__} {exc}) : relis tes assert.")
    else:
        missed = []
        for faulty in (solve_linear_wrong_sign, solve_linear_divides_by_zero, solve_linear_forgets_all):
            try:
                check_solve_linear(faulty)
            except Exception:  # noqa: BLE001 - an AssertionError or any other error: the fault is caught
                continue
            missed.append(faulty.__name__)
        if missed:
            print("❌ Ex R2.28 : ton test passe sur une version fausse :", ", ".join(missed),
                  "; ajoute le cas qu'elle rate.")
        else:
            print("✅ Ex R2.28 : ton test passe sur la fonction juste et attrape les trois versions fausses.")

## Partie C · Systèmes de deux équations à deux inconnues

Fiche §7. La formule de Cramer programmée, les deux droites d'un système, trois systèmes à prédire, et un système presque dégénéré.

### Ex R2.21 — Système 2 × 2 par la formule de Cramer · `solve_2x2` 🔨 ★★ ⏱️ 40 min
**Objectif :** programmer la formule de Cramer, et refuser un système de déterminant nul.  
**Prérequis :** fiche §7, « Par la formule de Cramer » ; workbookIA 0A §100.6.2 (`raise`) · **mymath :** `algebra.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en R1). Ouvre ta copie `mon_travail/mymath/algebra.py`, lis la docstring du module **et** celle de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. N'utilise ni SymPy ni NumPy dans `algebra.py` : les tests s'en servent pour vérifier tes résultats.

Écris `solve_2x2(A, b)` : `A` contient les coefficients ligne par ligne, `[[a, b], [c, d]]`, et `b` les seconds membres, `[e, f]` (attention : dans le code, `b` est la liste des seconds membres, pas le coefficient ; dépaquette avec `(a11, a12), (a21, a22) = A`).
1. Calcule le déterminant $D = ad - bc$.
2. Si $D = 0$, lève une `ValueError` avec un message clair : le système n'a pas **une seule** solution.
3. Sinon, calcule $x$ et $y$ par la formule de Cramer, et rends le **tuple** `(x, y)`.

La vérification calcule `solve_2x2([[3, 4], [5, -2]], [5.5, 13.5])`, puis le nom de l'erreur levée pour un système de déterminant nul (avec `raised`, de la cellule des outils).

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R2")  # reload your saved file

In [ ]:
with wbm.attempt("R2.21a"):
    wbm.check("R2.21a", list(mymath.algebra.solve_2x2([[3, 4], [5, -2]], [5.5, 13.5])), computed=True)
with wbm.attempt("R2.21b"):
    wbm.check("R2.21b", raised(mymath.algebra.solve_2x2, [[2, -5], [-4, 10]], [1, 3]), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r2_algebra.py", "test_solve_2x2_")

### Ex R2.26 — Un système, deux droites : voir la solution 🎨 ★★ ⏱️ 20 min
**Objectif :** tracer les deux droites d'un système et lire sa solution, dans les trois cas.  
**Prérequis :** fiche §7, « Le regard géométrique » ; R0 (matplotlib) · **Parcours :** C, E

Complète `plot_system(A, b, ax=None)`, qui dessine le système $a_1 x + b_1 y = c_1$, $a_2 x + b_2 y = c_2$ (avec `A = [[a1, b1], [a2, b2]]` et `b = [c1, c2]`) :
- chaque équation est une droite, tracée pour $x$ de $-5$ à $5$ : si son coefficient de $y$ n'est pas nul, $y = \frac{c - a x}{b}$ ; sinon, la droite est **verticale**, $x = \frac{c}{a}$ (`ax.axvline`) ;
- si le déterminant n'est pas nul, marque le point commun (calculé avec `np.linalg.solve(A, b)`) et écris ses coordonnées dans le titre ;
- sinon, le titre dit « une infinité de solutions » quand les droites sont confondues, ce qui arrive quand, en plus, $a_1 c_2 - a_2 c_1 = 0$ et $b_1 c_2 - b_2 c_1 = 0$ (R2.15), et « aucune solution » dans l'autre cas ;
- une légende avec les deux équations, une grille, les axes nommés, une fenêtre $[-5 ; 5] \times [-5 ; 5]$ ;
- si on lui donne `ax` (une case d'une figure à plusieurs graphiques), elle dessine dedans ; sinon, elle crée sa figure (`fig, ax = plt.subplots()`) et l'affiche.

La vérification dessine $\begin{cases} x + 3y = 5 \\ 2x = -2 \end{cases}$ (une droite verticale). Lis la solution sur ta figure, note-la dans ta copie, et compare ta figure à celle de `05_solutions.ipynb`. R2.24 se servira de ta fonction.

In [ ]:
def plot_system(A, b, ax=None):
    """Draw the two lines of the system A·(x, y) = b, and their common point when there is exactly one."""
    raise NotImplementedError("trace les deux droites, puis le point commun")

In [ ]:
with wbm.attempt("R2.26"):
    plot_system([[1, 3], [2, 0]], [5, -2])

### Ex R2.24 — Prédire le nombre de solutions de trois systèmes, puis tracer leurs droites 🔮 ★ ⏱️ 15 min
**Objectif :** prévoir le nombre de solutions d'un système en regardant ses coefficients, puis le voir sur les droites.  
**Prérequis :** fiche §7 ; R2.26 (ta fonction `plot_system`) · **Parcours :** C, E

Trois systèmes :

$$S_1 : \begin{cases} 5x + y = 2 \\ 10x + 2y = 7 \end{cases} \qquad S_2 : \begin{cases} x + 2y = 4 \\ 3x - y = 5 \end{cases} \qquad S_3 : \begin{cases} 3x - 2y = 1 \\ -6x + 4y = -2 \end{cases}$$

**Avant d'exécuter quoi que ce soit**, et sans les résoudre, prédis le nombre de solutions de chacun : 1 pour aucune, 2 pour une seule, 3 pour une infinité. Regarde les coefficients de $x$ et de $y$ d'une équation à l'autre, puis les seconds membres (fiche §7). Écris tes trois prédictions, vérifie-les, puis lance l'expérience : elle calcule les trois déterminants et trace les trois systèmes avec ta fonction `plot_system` (R2.26).

In [ ]:
# The three systems of R2.24: name -> (A, b), for a1·x + b1·y = c1, a2·x + b2·y = c2
SYSTEMS_24 = {
    "S1": ([[5, 1], [10, 2]], [2, 7]),
    "S2": ([[1, 2], [3, -1]], [4, 5]),
    "S3": ([[3, -2], [-6, 4]], [1, -2]),
}

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_R2_24a = ...  # S1: 1, 2 or 3, written BEFORE running the experiment below
prediction_R2_24b = ...  # S2: 1, 2 or 3
prediction_R2_24c = ...  # S3: 1, 2 or 3

In [ ]:
for letter, prediction in zip("abc", [prediction_R2_24a, prediction_R2_24b, prediction_R2_24c]):
    wbm.check(f"R2.24{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes trois prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_R2_24a, prediction_R2_24b, prediction_R2_24c]):
    print("⏳ Écris d'abord tes trois prédictions (1, 2 ou 3), puis exécute à nouveau.")
else:
    for name, (A, b) in SYSTEMS_24.items():
        (a1, b1), (a2, b2) = A
        print(f"{name} : déterminant D = ad − bc = {a1 * b2 - b1 * a2}")
    with wbm.attempt("R2.24 figure"):
        fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
        for ax, (name, (A, b)) in zip(axes, SYSTEMS_24.items()):
            plot_system(A, b, ax=ax)
            ax.set_title(f"{name} : {ax.get_title()}", fontsize=10)
        fig.tight_layout()
        plt.show()

### Ex R2.25 — Expérience : un système presque dégénéré, un coefficient bouge de 0,001 et la solution s'envole 🔬 ★★ ⏱️ 20 min
**Objectif :** voir qu'un système de déterminant presque nul est très sensible à ses coefficients.  
**Prérequis :** fiche §7 ; R2.24 · **Parcours :** C

On étudie le système $\begin{cases} x + y = 2 \\ x + c\,y = 2{,}001 \end{cases}$ pour des valeurs de $c$ proches de 1. Ses deux droites sont presque parallèles.

- Écris `solve_near(c)` : la solution `[x, y]`, calculée avec `np.linalg.solve`.
- a) Sa solution pour $c = 1{,}001$.
- b) Sa solution pour $c = 1{,}002$ : le coefficient n'a bougé que de 0,001.
- c) Son déterminant pour $c = 1{,}001$ (calcule-le dans ta tête, puis en Python).
- d) La valeur de $y$ pour $c = 1{,}0001$.

Puis lance l'expérience : elle trace $y$ en fonction de $c - 1$, de $10^{-4}$ à $10^{-2}$, sur deux échelles logarithmiques. Dans ta copie, dis en une phrase ce que la courbe montre, et pourquoi un déterminant presque nul est un signal d'alarme.

In [ ]:
def solve_near(c):
    """Solution [x, y] of x + y = 2, x + c·y = 2.001 (with np.linalg.solve)."""
    raise NotImplementedError("pose la matrice A et le second membre, puis appelle np.linalg.solve")


answer_R2_25c = ...  # c) the determinant for c = 1.001 (a number)

In [ ]:
with wbm.attempt("R2.25a"):
    wbm.check("R2.25a", list(solve_near(1.001)), computed=True)
with wbm.attempt("R2.25b"):
    wbm.check("R2.25b", list(solve_near(1.002)), computed=True)
wbm.check("R2.25c", answer_R2_25c)
with wbm.attempt("R2.25d"):
    wbm.check("R2.25d", solve_near(1.0001)[1], computed=True)

**L'expérience** : $y$ en fonction de $c - 1$.

In [ ]:
with wbm.attempt("R2.25 figure"):
    gaps = np.logspace(-4, -2, 60)                    # c − 1, from 0.0001 to 0.01
    ys = [solve_near(1 + gap)[1] for gap in gaps]
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.loglog(gaps, ys, "o-", markersize=3)
    ax.set_xlabel("c − 1, le déterminant (échelle logarithmique)")
    ax.set_ylabel("y, la solution (échelle logarithmique)")
    ax.set_title("Quand le déterminant tend vers 0, la solution s'envole")
    ax.grid(True, which="both", alpha=0.3)
    plt.show()

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Développes-tu et factorises-tu sans erreur de signe, identités remarquables comprises, et vérifies-tu ton résultat (valeur test ou SymPy) ?
2. Résous-tu une équation ou une inéquation du premier degré, une équation produit nul et une valeur absolue, en écrivant l'ensemble des solutions (intervalles compris) ?
3. Résous-tu un système 2 × 2 de trois façons, et sais-tu dire, avec le déterminant, s'il a une seule solution ?

**La suite** : R3, *Fonctions*. Dans workbookIA, continue le chapitre 0A au rythme de ton tableau de bord.